In [0]:
# === KONFIGURACJA ===
dbutils.widgets.text("projekt", "inspektor_budzet")
dbutils.widgets.text("dostawca", "rowkop")

projekt = dbutils.widgets.get("projekt")
dostawca = dbutils.widgets.get("dostawca")

catalog = projekt
schema = dostawca

print(f"Notebook 03 — Gold")
print(f"Dostawca: {dostawca}")



In [0]:
# === WCZYTANIE DANYCH ===

# Dane z ERP gminy (przetworzone przez Silver)
df_erp = spark.table(f"{catalog}.{schema}.silver_summary")

# Dane z faktury dostawcy (sparsowane z PDF w Bronze)
df_faktura = spark.table(f"{catalog}.{schema}.bronze_faktura")

print(f"ERP — liczba pozycji: {df_erp.count()}")
print(f"Faktura — liczba pozycji: {df_faktura.count()}")

display(df_erp)
display(df_faktura)

In [0]:
# === POŁĄCZENIE DANYCH ERP I FAKTURY ===
# JOIN po position_id — każda pozycja kontraktu w jednym wierszu
df_joined = df_erp.join(df_faktura, on="position_id", how="inner")

display(df_joined)

In [0]:
from pyspark.sql import functions as F

# === OBLICZENIA KWOT ===

# Reguła tiered dla wykopu — wg § 3.2 kontraktu
# Pierwsze 500mb po 45 zł, każde kolejne po 40,50 zł
def kwota_erp(position_id, erp_ilosc, faktura_cena_netto):
    if position_id == "WYKOP_ROWU":
        if erp_ilosc <= 500:
            return erp_ilosc * 45.0
        else:
            return 500 * 45.0 + (erp_ilosc - 500) * 40.50
    else:
        return erp_ilosc * faktura_cena_netto

# Obliczamy kwoty i rozbieżności dla każdej pozycji
rows_gold = []
for row in df_joined.collect():
    erp_kwota = kwota_erp(row.position_id, row.erp_ilosc, row.faktura_cena_netto)
    roznica_ilosc = row.faktura_ilosc - row.erp_ilosc
    roznica_kwota = row.faktura_wartosc_netto - erp_kwota
    status = "OK" if roznica_kwota == 0 else "NIEZGODNOŚĆ"

    rows_gold.append((
        row.position_id,
        float(row.erp_ilosc),
        float(row.faktura_ilosc),
        float(roznica_ilosc),
        float(erp_kwota),
        float(row.faktura_wartosc_netto),
        float(roznica_kwota),
        status
    ))

df_gold = spark.createDataFrame(rows_gold, [
    "position_id",
    "erp_ilosc",
    "faktura_ilosc",
    "roznica_ilosc",
    "erp_kwota_netto",
    "faktura_kwota_netto",
    "roznica_kwota",
    "status"
])

display(df_gold)


In [0]:
# === ZAPIS RAPORTU ===
# Zapisujemy raport rozbieżności jako tabelę Gold
df_gold.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.gold_raport_rozbieznosci")
print(f"Zapisano: {catalog}.{schema}.gold_raport_rozbieznosci ✅")